# Shortest Path in Dhaka: A* vs Bidirectional Dijkstra vs Hill Climbing
**Idea:** the road map is a graph. Every road junction is a *node*, every road piece is an *edge* weighted by its length in metres. We find a route from a source to a destination with three algorithms and compare them.

| Step | What happens |
|---|---|
| 1 | Get the OpenStreetMap dataset (`.osm.pbf`) into Colab |
| 2 | Build a road graph from it |
| 3 | Snap source and destination coordinates to the nearest junction |
| 4 | Implement the three algorithms |
| 5 | Run them on one route and draw it on the map |
| 6 | Test them on many random routes to decide fairly which is best |

In [ ]:
!pip install -q pyrosm networkx folium scipy pandas numpy matplotlib scikit-learn tqdm psutil

## Step 1: Get the dataset
Pick **one** option by changing `DATA_SOURCE`:
- `"upload"`: upload the `.osm.pbf` file from your computer (a 340 MB file can take a few minutes)
- `"drive"`: put the file in Google Drive first, then set `DRIVE_PATH`
- `"download"`: download from the Geofabrik link your faculty gave

In [ ]:
import os

DATA_SOURCE = "download"        # "upload", "drive" or "download"
DRIVE_PATH  = "/content/drive/MyDrive/bangladesh-latest.osm.pbf"   # only for "drive"
DATASET_URL = "https://download.geofabrik.de/asia/bangladesh-latest.osm.pbf"

if DATA_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    PBF_FILE = "/content/" + list(uploaded.keys())[0]
elif DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PBF_FILE = DRIVE_PATH
else:
    import requests
    PBF_FILE = "/content/bangladesh-latest.osm.pbf"
    if not os.path.exists(PBF_FILE):
        print("Downloading dataset...")
        with requests.get(DATASET_URL, stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(PBF_FILE, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    f.write(chunk)

print("Dataset:", PBF_FILE)
print("Size   :", round(os.path.getsize(PBF_FILE) / 1024**2, 2), "MB")

## Step 2: Build the road graph
`pyrosm` reads the roads from the big file. We keep a box around Dhaka, then convert the roads into a directed `networkx` graph.

In [ ]:
import time, heapq, json, random
import numpy as np
import pandas as pd
import networkx as nx
import folium
import matplotlib.pyplot as plt
from pyrosm import OSM
from sklearn.neighbors import BallTree
from IPython.display import display

USE_DHAKA_ONLY = True
bbox = [90.30, 23.65, 90.55, 23.90]      # [west, south, east, north]

osm = OSM(PBF_FILE, bounding_box=bbox) if USE_DHAKA_ONLY else OSM(PBF_FILE)
nodes, edges = osm.get_network(network_type="driving", nodes=True)
print("Junctions (nodes):", len(nodes), "| Road pieces (edges):", len(edges))

In [ ]:
G = nx.DiGraph()

# each junction = a node with coordinates
for node_id, lon, lat in zip(nodes["id"], nodes["lon"], nodes["lat"]):
    G.add_node(node_id, x=float(lon), y=float(lat))

# each road piece = an edge, weight = length in metres
for u, v, length in zip(edges["u"], edges["v"], edges["length"]):
    if pd.isna(length) or not np.isfinite(length) or length <= 0:
        continue
    if G.has_edge(u, v):
        G[u][v]["weight"] = min(G[u][v]["weight"], float(length))
    else:
        G.add_edge(u, v, weight=float(length))

print("Graph ready:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

## Step 3: Choose source and destination
Give normal latitude/longitude. `find_nearest_node` snaps each point to the closest junction (fast lookup with a `BallTree`). **Change the four numbers to try other places** (right-click a spot in Google Maps to copy its coordinates).

In [ ]:
node_ids = list(G.nodes())
coords_rad = np.radians(np.array([[G.nodes[n]["y"], G.nodes[n]["x"]] for n in node_ids]))
tree = BallTree(coords_rad, metric="haversine")

def find_nearest_node(lat, lon):
    _, idx = tree.query(np.radians([[lat, lon]]), k=1)
    return node_ids[idx[0][0]]

# ----- CHANGE THESE -----
source_lat, source_lon = 23.8223, 90.3654            # example: Mirpur
destination_lat, destination_lon = 23.7330, 90.3925  # example: Old Dhaka
# ------------------------

source = find_nearest_node(source_lat, source_lon)
destination = find_nearest_node(destination_lat, destination_lon)
print("Source node     :", source, (G.nodes[source]["y"], G.nodes[source]["x"]))
print("Destination node:", destination, (G.nodes[destination]["y"], G.nodes[destination]["x"]))

## Step 4: The three algorithms

### A* Search
For each node it tracks `f = g + h`, where `g` is the distance travelled so far and `h` is the straight-line distance to the destination (a guess of what is left). It always expands the lowest `f`, so it heads toward the goal instead of spreading everywhere. The straight line never overestimates the real road distance, so A* returns the **shortest** path. The route is rebuilt by following `parent` links backwards.

In [ ]:
EARTH_RADIUS = 6371000

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = np.radians([lat1, lon1, lat2, lon2])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 2 * EARTH_RADIUS * np.arcsin(np.sqrt(a))

def heuristic(node, goal):
    """Straight-line distance (metres) from node to goal."""
    return haversine(G.nodes[node]["y"], G.nodes[node]["x"],
                     G.nodes[goal]["y"], G.nodes[goal]["x"])

def astar_search(start, goal):
    t0 = time.perf_counter()
    open_list = [(heuristic(start, goal), start)]     # (f = g + h, node)
    g_score, parent, closed, explored = {start: 0.0}, {}, set(), 0
    while open_list:
        _, current = heapq.heappop(open_list)
        if current in closed:
            continue
        closed.add(current); explored += 1
        if current == goal:
            path = [goal]
            while path[-1] != start:
                path.append(parent[path[-1]])
            path.reverse()
            return {"path": path, "distance": g_score[goal], "nodes_explored": explored,
                    "execution_time": time.perf_counter() - t0}
        for nb in G.successors(current):
            if nb in closed:
                continue
            new_g = g_score[current] + G[current][nb]["weight"]
            if new_g < g_score.get(nb, float("inf")):
                g_score[nb] = new_g
                parent[nb] = current
                heapq.heappush(open_list, (new_g + heuristic(nb, goal), nb))
    return {"path": None, "distance": float("inf"), "nodes_explored": explored,
            "execution_time": time.perf_counter() - t0}

### Bidirectional Dijkstra
Plain Dijkstra spreads out evenly from the source like a ripple. Bidirectional Dijkstra runs **two ripples**: one forward from the source, one backward from the destination (along reversed roads). When they meet and no shorter meeting is possible, the shortest path is the left half joined to the right half. It is **optimal** and needs no heuristic, but each ripple spreads in all directions, so it usually explores more nodes than A*.

### Hill Climbing
From the current node, it moves to the unvisited neighbour that is **closest to the destination** and never goes back. It is very fast and uses almost no memory, but it only looks one step ahead. On real roads (dead ends, one-way streets, rivers) it can get **stuck** and fail, and when it does reach the goal the route is usually not the shortest.

In [ ]:
def bidirectional_dijkstra(start, goal):
    t0 = time.perf_counter()
    if start == goal:
        return {"path": [start], "distance": 0.0, "nodes_explored": 1,
                "execution_time": time.perf_counter() - t0}

    reverse_G = G.reverse(copy=False)          # same roads, directions flipped (for the backward search)
    dist_f, dist_b = {start: 0.0}, {goal: 0.0}
    parent_f, parent_b = {}, {}
    pq_f, pq_b = [(0.0, start)], [(0.0, goal)]
    settled_f, settled_b = set(), set()
    best_distance, meeting_node, explored = float("inf"), None, 0

    while pq_f and pq_b:
        while pq_f and pq_f[0][1] in settled_f: heapq.heappop(pq_f)
        while pq_b and pq_b[0][1] in settled_b: heapq.heappop(pq_b)
        if not pq_f or not pq_b:
            break
        # stop when the two searches have provably met on the shortest path
        if pq_f[0][0] + pq_b[0][0] >= best_distance:
            break

        if pq_f[0][0] <= pq_b[0][0]:           # expand the FORWARD search (from source)
            d, current = heapq.heappop(pq_f)
            if current in settled_f: continue
            settled_f.add(current); explored += 1
            for nb in G.successors(current):
                nd = d + G[current][nb]["weight"]
                if nd < dist_f.get(nb, float("inf")):
                    dist_f[nb] = nd; parent_f[nb] = current
                    heapq.heappush(pq_f, (nd, nb))
                    if nb in dist_b and nd + dist_b[nb] < best_distance:
                        best_distance, meeting_node = nd + dist_b[nb], nb
            if current in dist_b and dist_f[current] + dist_b[current] < best_distance:
                best_distance, meeting_node = dist_f[current] + dist_b[current], current
        else:                                  # expand the BACKWARD search (from destination)
            d, current = heapq.heappop(pq_b)
            if current in settled_b: continue
            settled_b.add(current); explored += 1
            for nb in reverse_G.successors(current):
                nd = d + reverse_G[current][nb]["weight"]
                if nd < dist_b.get(nb, float("inf")):
                    dist_b[nb] = nd; parent_b[nb] = current
                    heapq.heappush(pq_b, (nd, nb))
                    if nb in dist_f and nd + dist_f[nb] < best_distance:
                        best_distance, meeting_node = nd + dist_f[nb], nb
            if current in dist_f and dist_f[current] + dist_b[current] < best_distance:
                best_distance, meeting_node = dist_f[current] + dist_b[current], current

    if meeting_node is None:
        return {"path": None, "distance": float("inf"), "nodes_explored": explored,
                "execution_time": time.perf_counter() - t0}

    left = [meeting_node]                      # source -> meeting point
    while left[-1] != start:
        left.append(parent_f[left[-1]])
    left.reverse()
    right, cur = [], meeting_node              # meeting point -> destination
    while cur != goal:
        cur = parent_b[cur]
        right.append(cur)
    return {"path": left + right, "distance": best_distance, "nodes_explored": explored,
            "execution_time": time.perf_counter() - t0}


def hill_climbing(start, goal):
    """Always step to the unvisited neighbour that is closest (straight-line) to the goal.
    Never goes back, so it can get stuck in a dead end."""
    t0 = time.perf_counter()
    current, path, visited, explored = start, [start], {start}, 0
    while current != goal:
        explored += 1
        best_nb, best_h = None, float("inf")
        for nb in G.successors(current):
            if nb in visited:
                continue
            h = heuristic(nb, goal)
            if h < best_h:
                best_h, best_nb = h, nb
        if best_nb is None:                    # dead end -> hill climbing gives up
            break
        current = best_nb
        visited.add(current); path.append(current)

    elapsed = time.perf_counter() - t0
    if current != goal:
        return {"path": None, "partial_path": path, "distance": float("inf"),
                "nodes_explored": explored, "execution_time": elapsed}
    dist = sum(G[path[i]][path[i+1]]["weight"] for i in range(len(path) - 1))
    return {"path": path, "distance": dist, "nodes_explored": explored, "execution_time": elapsed}

## Step 5: Run all three on this route

In [ ]:
algorithms = {
    "A* Search": astar_search,
    "Bidirectional Dijkstra": bidirectional_dijkstra,
    "Hill Climbing": hill_climbing,
}

results = {}
for name, algo in algorithms.items():
    r = algo(source, destination)
    results[name] = r
    ok = r["path"] is not None
    print(f"{name:24s} {'Success' if ok else 'FAILED '} | "
          f"{(str(round(r['distance']/1000, 3)) + ' km') if ok else 'no route':>10s} | "
          f"{r['execution_time']:.4f} s | {r['nodes_explored']} nodes explored")

comparison_df = pd.DataFrame([{
    "Algorithm": n,
    "Status": "Success" if r["path"] is not None else "Failed",
    "Distance (km)": r["distance"] / 1000 if r["path"] is not None else np.nan,
    "Execution Time (sec)": r["execution_time"],
    "Nodes Explored": r["nodes_explored"],
    "Path Nodes": len(r["path"]) if r["path"] is not None else 0,
} for n, r in results.items()])
display(comparison_df.round(4))

### See the routes on the map
Each algorithm now gets its **own separate map**. Each path is a list of node ids; we look up each node's latitude/longitude and draw them as a line (`folium.PolyLine`).

| Map | Colour | File saved |
|---|---|---|
| A* Search | blue | `map_astar.html` |
| Bidirectional Dijkstra | purple | `map_bidirectional_dijkstra.html` |
| Hill Climbing | orange (or red dashed if it gets stuck) | `map_hill_climbing.html` |

If Hill Climbing fails, its **partial path** is drawn as a dashed red line ending at the dead end where it got stuck. Each map also has a small box showing that algorithm's distance, time and nodes explored.

*The tiles come from Esri and OpenStreetMap-France, which need no API key. The default OpenStreetMap tile server blocks Colab and previews, which caused the "permission" error.*

In [ ]:
def make_base_map():
    """Create a fresh map with the tile layers and the source/destination markers."""
    center_lat = (G.nodes[source]["y"] + G.nodes[destination]["y"]) / 2
    center_lon = (G.nodes[source]["x"] + G.nodes[destination]["x"]) / 2
    m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles=None)
    folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
                     attr="Tiles &copy; Esri", name="Esri Street", max_zoom=19).add_to(m)
    folium.TileLayer(tiles="https://{s}.tile.openstreetmap.fr/osmfr/{z}/{x}/{y}.png",
                     attr="&copy; OpenStreetMap contributors", name="OSM France", max_zoom=19).add_to(m)
    folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                     attr="Tiles &copy; Esri", name="Satellite", max_zoom=19).add_to(m)
    folium.Marker([G.nodes[source]["y"], G.nodes[source]["x"]],
                  popup="Source", icon=folium.Icon(color="green")).add_to(m)
    folium.Marker([G.nodes[destination]["y"], G.nodes[destination]["x"]],
                  popup="Destination", icon=folium.Icon(color="red")).add_to(m)
    return m


def draw_algorithm_map(name, color, filename):
    """Draw ONE algorithm's route on its own map, save it as HTML and show it."""
    r = results[name]
    m = make_base_map()
    success = r["path"] is not None

    if success:
        pts = [[G.nodes[n]["y"], G.nodes[n]["x"]] for n in r["path"]]
        folium.PolyLine(pts, color=color, weight=6, opacity=0.9,
                        tooltip=f"{name} | {r['distance']/1000:.2f} km").add_to(m)
        status = f"Distance: <b>{r['distance']/1000:.2f} km</b>"
    else:
        # failed (only Hill Climbing can fail): draw the partial path up to the dead end
        pts = [[G.nodes[n]["y"], G.nodes[n]["x"]] for n in r.get("partial_path", [])]
        if pts:
            folium.PolyLine(pts, color="red", weight=4, opacity=0.9, dash_array="4 8",
                            tooltip=f"{name} got stuck after {len(pts)} nodes").add_to(m)
            folium.CircleMarker(pts[-1], radius=8, color="black", fill=True, fill_color="red",
                                popup=f"{name} stuck here (dead end)").add_to(m)
        status = "<b style='color:red;'>FAILED</b> (stuck at a dead end)"

    # small info box on the map
    info = f"""
    <div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999; background: white;
                padding: 10px 14px; border: 2px solid #888; border-radius: 6px; font-size: 14px;">
    <b style="color:{color if success else 'red'};">{name}</b><br>
    {status}<br>
    Time: {r['execution_time']:.4f} s<br>
    Nodes explored: {r['nodes_explored']}
    </div>"""
    m.get_root().html.add_child(folium.Element(info))

    if pts:
        m.fit_bounds(pts)
    folium.LayerControl().add_to(m)
    m.save(filename)          # open this file in a browser if the preview is blank
    print(f"{name}: map saved to {filename}")
    return m


# One map per algorithm
map_astar = draw_algorithm_map("A* Search", "blue", "map_astar.html")
display(map_astar)

In [ ]:
map_bidir = draw_algorithm_map("Bidirectional Dijkstra", "purple", "map_bidirectional_dijkstra.html")
display(map_bidir)

In [ ]:
# Hill Climbing: orange if it reaches the goal, red dashed (partial path) if it gets stuck
map_hill = draw_algorithm_map("Hill Climbing", "orange", "map_hill_climbing.html")
display(map_hill)

same = (results["A* Search"]["path"] is not None
        and results["A* Search"]["path"] == results["Bidirectional Dijkstra"]["path"])
print("A* and Bidirectional Dijkstra found the IDENTICAL road sequence:", same)

## Step 6: Which algorithm is better? Test on many routes
One route is not enough to decide. Here we test all three on **many random routes** (at least `MIN_KM` apart) and measure:
- **Success rate**: how often it found any route
- **Extra distance**: how much longer than the true shortest path (A* distance) the route is
- **Time** and **nodes explored**: how much work it did

Random points are chosen only from the *largest connected part* of the road network, so every route has a valid answer.

In [ ]:
N_TRIALS = 20        # number of random routes
MIN_KM   = 3         # minimum straight-line distance between source and destination
rng = random.Random(42)

# keep only junctions that can all reach each other (so a route always exists)
main_part = list(max(nx.strongly_connected_components(G), key=len))

pairs = []
while len(pairs) < N_TRIALS:
    s, t = rng.sample(main_part, 2)
    if heuristic(s, t) >= MIN_KM * 1000:
        pairs.append((s, t))

rows = []
for i, (s, t) in enumerate(pairs, 1):
    for name, algo in algorithms.items():
        r = algo(s, t)
        rows.append({"trial": i, "algorithm": name,
                     "success": r["path"] is not None,
                     "distance_km": r["distance"] / 1000 if r["path"] is not None else np.nan,
                     "time_s": r["execution_time"],
                     "nodes": r["nodes_explored"]})
trials = pd.DataFrame(rows)

# true shortest distance per trial (A* is optimal)
best = trials[trials.algorithm == "A* Search"].set_index("trial")["distance_km"]
trials["extra_pct"] = (trials["distance_km"] / trials["trial"].map(best) - 1) * 100

summary_df = trials.groupby("algorithm").agg(
    success_rate_pct=("success", lambda x: 100 * x.mean()),
    avg_extra_distance_pct=("extra_pct", "mean"),
    avg_time_s=("time_s", "mean"),
    avg_nodes_explored=("nodes", "mean"),
).round(3)
print(f"Results over {N_TRIALS} random routes:")
display(summary_df)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
colors = ["tab:blue", "tab:purple", "tab:orange"]
order = ["A* Search", "Bidirectional Dijkstra", "Hill Climbing"]
plots = [("success_rate_pct", "Success rate (%)"),
         ("avg_extra_distance_pct", "Avg extra distance vs shortest (%)"),
         ("avg_time_s", "Avg time (s)"),
         ("avg_nodes_explored", "Avg nodes explored")]
for ax, (col, title) in zip(axes, plots):
    ax.bar(order, summary_df.loc[order, col], color=colors)
    ax.set_title(title); ax.tick_params(axis="x", rotation=20); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Automatic conclusion
s = summary_df
optimal = s[(s.success_rate_pct == 100) & (s.avg_extra_distance_pct.abs() < 1e-6)]
print("CONCLUSION")
print("-" * 60)
print("Algorithms that always found the shortest path:", ", ".join(optimal.index))
if len(optimal):
    fastest = optimal["avg_time_s"].idxmin()
    fewest  = optimal["avg_nodes_explored"].idxmin()
    print("Fastest of those      :", fastest)
    print("Fewest nodes explored :", fewest)
hc = s.loc["Hill Climbing"]
print(f"Hill Climbing: succeeded on {hc.success_rate_pct:.0f}% of routes", end="")
if hc.success_rate_pct > 0:
    print(f", and its routes were on average {hc.avg_extra_distance_pct:.1f}% longer than the shortest.")
else:
    print(".")

In [ ]:
comparison_df.to_csv("shortest_path_comparison.csv", index=False)
summary_df.to_csv("multi_route_summary.csv")
trials.to_csv("multi_route_trials.csv", index=False)

summary = {n: {
    "distance_km": r["distance"] / 1000 if r["path"] is not None else None,
    "execution_time": r["execution_time"],
    "nodes_explored": r["nodes_explored"],
    "path_nodes": len(r["path"]) if r["path"] is not None else 0} for n, r in results.items()}
with open("algorithm_results.json", "w") as f:
    json.dump(summary, f, indent=4)

from google.colab import files
for fn in ["shortest_path_comparison.csv", "multi_route_summary.csv", "multi_route_trials.csv",
           "map_astar.html", "map_bidirectional_dijkstra.html", "map_hill_climbing.html", "algorithm_results.json"]:
    files.download(fn)